# K2 training (EXPLORATORY)
One fresh neutral or contrary adapter per run. Default is CPU dry-run; no downloads or secrets are needed. Attach a local Gemma 3 1B or 4B instruction snapshot and reviewed repo dataset for a future owner-run GPU session. Never attach Study D adapters.


In [ ]:
from pathlib import Path
import sys, json, hashlib, os
DRY_RUN = True
MODE = 'train'
REPO = Path('/kaggle/input/taa-k2-repo/taa-lab')
# For local CPU dry-run only, set REPO to the checkout root.
PINNED_COMMIT = ''  # Full reviewed K2 commit; dataset metadata must match.
MODEL = 'gemma-3-1b-it'  # or gemma-3-4b-it
MODEL_PATH = Path('/kaggle/input/gemma-model/transformers/default/1')
MODEL_REVISION = ''  # Exact upstream snapshot revision, recorded by Chris.
OUTPUT = Path('/kaggle/working/k2-output')
if not DRY_RUN and (len(PINNED_COMMIT)!=40 or len(MODEL_REVISION)!=40):
    raise ValueError('Pin repo and model revisions before running')
if not DRY_RUN:
    assert (REPO/'COMMIT.txt').read_text().strip()==PINNED_COMMIT
sys.path.insert(0,str(REPO))
from exploratory.k2_kaggle import core
core.validate_assets()
OUTPUT.mkdir(parents=True,exist_ok=False)
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'
metadata = dict(dry_run=DRY_RUN, repo_commit=PINNED_COMMIT, model_id=core.model_id(MODEL), model_revision=MODEL_REVISION,
                asset_hashes=core.validate_assets())
def retain(row):
    with (OUTPUT/'events.jsonl').open('a',encoding='utf-8') as f:
        f.write(json.dumps(row,ensure_ascii=False)+'\n')
ARM='neutral'  # neutral or contrary; base needs no training
TRAIN_SEED=247056
assert ARM in ('neutral','contrary')
metadata.update(arm=ARM,training_seed=TRAIN_SEED)
records=[json.loads(s) for s in (core.ASSETS/(ARM+'-2000-train.jsonl')).read_text().splitlines()]
order=json.loads((core.ASSETS/'training-order.json').read_text())['high']['record_ids']
assert len(records)==len(order)==2000

SESSION_HOURS=8.5
import time
DEADLINE=time.monotonic()+SESSION_HOURS*3600
metadata.update(mode=MODE,recipe_sha256=core.digest(core.RECIPE),implementation_sha256=hashlib.sha256(Path(core.__file__).read_bytes().replace(b'\r\n',b'\n')).hexdigest())
(OUTPUT/'metadata.json').write_text(json.dumps(metadata,indent=2))


In [ ]:
try:
    def qualify(model,tokenizer,data):
        from peft import PeftModel
        probe=records[0]['messages']
        (OUTPUT/'training-template.txt').write_text(core.prompt(tokenizer,probe[:-1],False))
        eval_messages=[dict(role='user',content=core.manager_cases()[0]['task'])] if 'task' in core.manager_cases()[0] else core.acquisition_prompts()[0]
        (OUTPUT/'eval-template.txt').write_text(core.prompt(tokenizer,eval_messages,False))
        (OUTPUT/'mask-probe.json').write_text(json.dumps(data[0]))
        base=model.unload()
        reloaded=PeftModel.from_pretrained(base,OUTPUT/'adapter',adapter_name='qualification',is_trainable=False)
        reloaded.load_adapter(OUTPUT/'adapter',adapter_name='reload_check',is_trainable=False)
        reloaded.set_adapter('reload_check')
        reloaded.set_adapter('qualification')
        resolved=reloaded.generation_config.to_dict()
        resolved.update(max_new_tokens=16,do_sample=False)
        metadata['qualification_generation_config']=resolved
        ids=torch.tensor([core.prompt(tokenizer,eval_messages)],device='cuda:0')
        started=time.monotonic()
        with torch.no_grad(): result=reloaded.generate(ids,max_new_tokens=16,do_sample=False)
        with reloaded.disable_adapter():
            with torch.no_grad(): reloaded.generate(ids,max_new_tokens=1,do_sample=False)
        retain(dict(event='qualification_probe',scientific=False,reply=tokenizer.decode(result[0].tolist()),
            peak_gpu_bytes=torch.cuda.max_memory_allocated(),tokens_per_second=(result.shape[-1]-ids.shape[-1])/(time.monotonic()-started)))
    if not DRY_RUN:
        metadata['packages']=core.check_stack()
        metadata['base_inventory_sha256']=core.digest(core.inventory(MODEL_PATH))
        import torch, transformers, peft
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
        metadata['gpu'] = torch.cuda.get_device_name(0)
        tokenizer=AutoTokenizer.from_pretrained(MODEL_PATH,local_files_only=True,trust_remote_code=False)
        model=core.load_model(MODEL,transformers,MODEL_PATH,
            quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16),device_map={'':0})
        metadata['loaded_class']=type(model).__name__
        metadata['resolved_generation_config']=model.generation_config.to_dict()
        def prompt(messages): return core.prompt(tokenizer,messages)
    if DRY_RUN:
        retain(dict(event='training_plan',model_id=core.model_id(MODEL),model_revision=MODEL_REVISION,examples=len(records),order=order,optimizer_steps=250,science=False))
    else:
        from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig
        from transformers import Trainer, TrainingArguments, TrainerCallback
        set_seed(TRAIN_SEED)
        model=prepare_model_for_kbit_training(model)
        model=get_peft_model(model,LoraConfig(r=8,lora_alpha=16,lora_dropout=0,bias='none',task_type='CAUSAL_LM',
            target_modules=core.lora_targets(MODEL,[n for n,m in model.named_modules()])))
        metadata['lora_modules']=[n for n,m in model.named_modules() if hasattr(m,'lora_A')]
        metadata['trainable_parameters']=sum(p.numel() for p in model.parameters() if p.requires_grad)
        data=[]
        for key in order:
            data.append(core.masked_record(tokenizer,records[int(key[1:])-1]['messages']))
        retain(dict(event='tokenization',examples=len(data),tokens=sum(len(x['input_ids']) for x in data),
                    supervised_tokens=sum(sum(t!=-100 for t in x['labels']) for x in data)))
        class FixedTrainer(Trainer):
            def _get_train_sampler(self,*args,**kwargs): return torch.utils.data.SequentialSampler(self.train_dataset)
        class Log(TrainerCallback):
            def on_log(self,args,state,control,logs=None,**kwargs):
                import math
                retain(dict(event='train_log',step=state.global_step,logs=logs))
                if logs and 'loss' in logs and not math.isfinite(logs['loss']): raise ValueError('nonfinite loss')
            def on_step_end(self,args,state,control,**kwargs):
                if state.global_step%25==0:
                    checkpoint=OUTPUT/('incomplete-step-'+str(state.global_step))
                    model.save_pretrained(checkpoint,safe_serialization=True)
                    (checkpoint/'INCOMPLETE.json').write_text(json.dumps(dict(scientific=False,step=state.global_step)))
                if time.monotonic()>=DEADLINE: raise TimeoutError('session deadline; no resume')
        def collate(batch): return {k:torch.tensor([r[k] for r in batch]) for k in batch[0]}
        args=TrainingArguments(output_dir=str(OUTPUT/'trainer'),num_train_epochs=1,learning_rate=.0002,
            per_device_train_batch_size=1,gradient_accumulation_steps=8,fp16=True,optim='adamw_torch',
            weight_decay=0,lr_scheduler_type='constant',warmup_steps=0,seed=TRAIN_SEED,data_seed=TRAIN_SEED,
            max_steps=5 if MODE=='qualify' else 250,save_strategy='no',logging_steps=1,report_to=[],gradient_checkpointing=True)
        trainer=FixedTrainer(model=model,args=args,train_dataset=data,data_collator=collate,callbacks=[Log()])
        try:
            trainer.train()
            assert trainer.state.global_step==(5 if MODE=='qualify' else 250)
            model.save_pretrained(OUTPUT/'adapter',safe_serialization=True)
            retain(dict(event='qualification_complete' if MODE=='qualify' else 'training_complete',optimizer_steps=trainer.state.global_step))
            if MODE=='qualify': qualify(model,tokenizer,data)
        except BaseException:
            model.save_pretrained(OUTPUT/'interrupted-adapter',safe_serialization=True)
            core.manifest(OUTPUT,metadata)
            raise
except BaseException as exc:
    retain(dict(event='run_failed',error=type(exc).__name__,message=str(exc)))
    raise
finally:
    (OUTPUT/'metadata.json').write_text(json.dumps(metadata,indent=2))
    core.manifest(OUTPUT,metadata)
